# 03 · Live win probability
Chance that the Hidden King side wins at each 3-minute snapshot, from team 0 minus team 1 gaps (net worth, kills, deaths, assists, level, net worth share), each also multiplied by minute / 30 because a lead counts for more later.
Model: `broadway_ml/models/live.py`. Split by match time; the snapshot a second before the end is never used.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from broadway_ml.flatten import load_tables, item_catalog
from broadway_ml import features as fe, evaluate as ev, export as ex
T = load_tables('../data/tables')
{k: len(v) for k, v in T.items()}
from broadway_ml.models import draft, live, builds, playstyle, rank
import os; os.makedirs('../models', exist_ok=True)
def plot_rel(table, ax=None, label='model'):
    ax = ax or plt.subplots(figsize=(5, 5))[1]
    ax.plot([0, 1], [0, 1], ls='--', c='grey', lw=1)
    ax.plot(table.predicted, table.observed, marker='o', label=label)
    ax.set(xlabel='predicted win chance', ylabel='observed win rate', xlim=(0, 1), ylim=(0, 1)); ax.legend()
    return ax

In [ ]:
rep = live.train(T, '../models')
rep['comparison']

In [ ]:
rep['auc_by_minute'].plot(title='AUC by minute', ylim=(0.5, 1.0), marker='o');

In [ ]:
plot_rel(rep['reliability'], label='live model');

**A curve for one match** (what the match page will draw):

In [ ]:
tl = live.live_frame(T['matches'], T['stats'])
mid = tl.match_id.iloc[-1]
import json
mdl = json.load(open('../models/live_win.json'))
g = tl[tl.match_id == mid]
p = [ex.score_linear(mdl, row) for row in g[mdl['features']].to_dict('records')]
plt.plot(g.minute, p, marker='o'); plt.axhline(.5, c='grey', ls='--'); plt.ylim(0, 1)
plt.title(f'match {mid}: Hidden King win chance (won: {bool(g.win.iloc[0])})');

**Next idea:** objectives (Walkers, Guardians, Patron phases, Mid Boss) via `include_objectives` in the pull are likely the strongest late-game signal.